# 🚗 Driver Monitoring System using YOLOv8

> **Note:** This is a college project and not implemented in actual car systems.

### Objectives
- Real-time monitoring of driver to check if he is distracted, using phone, or talking.
- Give warning whenever one of the mentioned things happens in real time.

---
## Table of Contents
1. Environment Setup
2. Dataset Download from Kaggle
3. Dataset Exploration & Preprocessing
4. YOLOv8 Model Training
5. Model Evaluation
6. Real-Time Detection & Inference
7. Export Model for Deployment

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install ultralytics kaggle opencv-python-headless matplotlib seaborn PyYAML Pillow

import os
import yaml
import shutil
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO

print('✅ All packages installed and imported successfully!')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Check if running on Google Colab or local
try:
    import google.colab
    IN_COLAB = True
    print('🟦 Running on Google Colab')
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/driver_monitoring'
except ImportError:
    IN_COLAB = False
    print('🖥️  Running on Local Machine')
    BASE_DIR = './driver_monitoring'

os.makedirs(BASE_DIR, exist_ok=True)
print(f'📁 Base directory: {BASE_DIR}')

## 2. Dataset Download from Kaggle

**Dataset:** [Driver Gaze RGB Dataset (LISA v1 & v2)](https://www.kaggle.com/datasets/yousefradwanlmao/driver-gaze-rgb-dataset-lisa-v1-v2)

### Setup Kaggle API
1. Go to [kaggle.com](https://www.kaggle.com) → Account → API → Create New Token
2. Download `kaggle.json`
3. Upload it below (Colab) or place it in `~/.kaggle/kaggle.json` (Local)

In [ ]:
# Setup Kaggle credentials
if IN_COLAB:
    from google.colab import files
    print('📂 Please upload your kaggle.json file:')
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('✅ Kaggle credentials configured!')
else:
    # For local - place kaggle.json manually at ~/.kaggle/kaggle.json
    kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_path):
        print('✅ Kaggle credentials found!')
    else:
        print('❌ Please place kaggle.json at ~/.kaggle/kaggle.json')
        print('   Download it from: kaggle.com → Account → API → Create New Token')

In [ ]:
# Download dataset directly from Kaggle
DATASET_DIR = os.path.join(BASE_DIR, 'dataset_raw')
os.makedirs(DATASET_DIR, exist_ok=True)

!kaggle datasets download -d yousefradwanlmao/driver-gaze-rgb-dataset-lisa-v1-v2 -p {DATASET_DIR} --unzip

print(f'\n📦 Dataset downloaded to: {DATASET_DIR}')
print('\nDirectory structure:')
for root, dirs, files_list in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:5]:
            print(f'{subindent}{f}')
        if len(files_list) > 5:
            print(f'{subindent}... and {len(files_list)-5} more')

## 3. Dataset Exploration & Preprocessing

We map gaze/behavior labels to our 4 distraction classes:
- `0` → **Attentive** (looking forward)
- `1` → **Distracted** (looking away)
- `2` → **Phone** (using phone)
- `3` → **Talking** (talking/eating)

In [ ]:
# Explore dataset structure
import glob

# Collect all image files
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
all_images = []
for ext in image_extensions:
    all_images.extend(glob.glob(os.path.join(DATASET_DIR, '**', ext), recursive=True))

print(f'📷 Total images found: {len(all_images)}')

# Check folder structure (categories may be subfolder names)
categories = set()
for img_path in all_images:
    parent = os.path.basename(os.path.dirname(img_path))
    categories.add(parent)

print(f'\n📂 Categories/folders found: {sorted(categories)}')

# Show sample images
sample_imgs = random.sample(all_images, min(8, len(all_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(os.path.basename(os.path.dirname(img_path)), fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Dataset Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Sample images displayed!')

In [ ]:
# Define class mapping
# Map dataset folder names to our 4 behavior classes
# Adjust this mapping based on the actual folder names you see above

CLASS_MAPPING = {
    # Add dataset folder names mapped to our class indices
    # Example mappings (update after exploring dataset):
    'forward': 0,          # Attentive
    'center': 0,           # Attentive
    'straight': 0,         # Attentive
    'left': 1,             # Distracted
    'right': 1,            # Distracted
    'mirror': 1,           # Distracted
    'phone': 2,            # Phone
    'texting': 2,          # Phone
    'talking': 3,          # Talking
    'eating': 3,           # Talking
}

CLASS_NAMES = {
    0: 'Attentive',
    1: 'Distracted',
    2: 'Phone',
    3: 'Talking'
}

CLASS_COLORS = {
    0: '#2ecc71',   # Green - Attentive
    1: '#f39c12',   # Orange - Distracted
    2: '#e74c3c',   # Red - Phone
    3: '#9b59b6',   # Purple - Talking
}

print('🏷️  Class Definitions:')
for cls_id, cls_name in CLASS_NAMES.items():
    print(f'  Class {cls_id}: {cls_name}')

In [ ]:
# Organize dataset into YOLOv8 format
# Structure: dataset/images/train|val|test/ and dataset/labels/train|val|test/

YOLO_DATASET_DIR = os.path.join(BASE_DIR, 'yolo_dataset')
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', split), exist_ok=True)

# Group images by category
categorized = {}
for img_path in all_images:
    folder = os.path.basename(os.path.dirname(img_path)).lower()
    # Find class ID
    cls_id = None
    for key, val in CLASS_MAPPING.items():
        if key in folder:
            cls_id = val
            break
    if cls_id is None:
        cls_id = 0  # Default to Attentive if unknown
    if cls_id not in categorized:
        categorized[cls_id] = []
    categorized[cls_id].append(img_path)

print('📊 Images per class:')
for cls_id, imgs in sorted(categorized.items()):
    print(f'  {CLASS_NAMES[cls_id]} (Class {cls_id}): {len(imgs)} images')

# Split: 70% train, 20% val, 10% test
TRAIN_RATIO, VAL_RATIO = 0.70, 0.20

split_counts = {'train': 0, 'val': 0, 'test': 0}

for cls_id, img_paths in categorized.items():
    random.shuffle(img_paths)
    n = len(img_paths)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    
    splits = {
        'train': img_paths[:n_train],
        'val':   img_paths[n_train:n_train+n_val],
        'test':  img_paths[n_train+n_val:]
    }
    
    for split_name, split_imgs in splits.items():
        for img_path in split_imgs:
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            
            # Save image
            filename = f'{cls_id}_{os.path.basename(img_path)}'
            dst_img = os.path.join(YOLO_DATASET_DIR, 'images', split_name, filename)
            img.save(dst_img)
            
            # Create YOLO label (full image bounding box as placeholder)
            # Format: class_id x_center y_center width height (normalized 0-1)
            label_filename = filename.rsplit('.', 1)[0] + '.txt'
            dst_label = os.path.join(YOLO_DATASET_DIR, 'labels', split_name, label_filename)
            with open(dst_label, 'w') as f:
                f.write(f'{cls_id} 0.5 0.5 1.0 1.0\n')
            
            split_counts[split_name] += 1

print(f'\n✅ Dataset split complete!')
print(f'  Train: {split_counts["train"]} images')
print(f'  Val:   {split_counts["val"]} images')
print(f'  Test:  {split_counts["test"]} images')

In [ ]:
# Create YOLOv8 data.yaml configuration file
data_yaml = {
    'path': os.path.abspath(YOLO_DATASET_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    4,
    'names': ['Attentive', 'Distracted', 'Phone', 'Talking']
}

DATA_YAML_PATH = os.path.join(BASE_DIR, 'data.yaml')
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('📝 data.yaml created:')
with open(DATA_YAML_PATH) as f:
    print(f.read())

In [ ]:
# Distribution chart
class_counts = {CLASS_NAMES[k]: len(v) for k, v in categorized.items()}
colors = [CLASS_COLORS[k] for k in sorted(categorized.keys())]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars = ax1.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='black', linewidth=0.5)
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Images')
ax1.set_xlabel('Behavior Class')
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')

ax2.pie(class_counts.values(), labels=class_counts.keys(), colors=colors,
        autopct='%1.1f%%', startangle=90, pctdistance=0.85)
ax2.set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. YOLOv8 Model Training

In [ ]:
# Initialize YOLOv8 model (pretrained on COCO)
# Options: yolov8n (nano), yolov8s (small), yolov8m (medium)
# For Google Colab use yolov8m; for local use yolov8n

MODEL_SIZE = 'yolov8m' if IN_COLAB else 'yolov8n'
model = YOLO(f'{MODEL_SIZE}.pt')
print(f'✅ Loaded {MODEL_SIZE} pretrained model')
print(model.info())

In [ ]:
# Training configuration
EPOCHS = 50         # Increase to 100 for better results
BATCH_SIZE = 16     # Reduce to 8 if running out of memory
IMAGE_SIZE = 640    # Standard YOLOv8 input size
PATIENCE = 10       # Early stopping patience

print(f'🚀 Starting YOLOv8 Training')
print(f'   Model:      {MODEL_SIZE}')
print(f'   Epochs:     {EPOCHS}')
print(f'   Batch:      {BATCH_SIZE}')
print(f'   Image size: {IMAGE_SIZE}')
print(f'   Dataset:    {DATA_YAML_PATH}')

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    name='driver_monitoring',
    project=os.path.join(BASE_DIR, 'runs'),
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    augment=True,
    mixup=0.1,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    verbose=True,
    save=True,
    plots=True,
    device=0 if not IN_COLAB else 0,  # GPU if available
)

print('\n✅ Training Complete!')

## 5. Model Evaluation

In [ ]:
# Load best trained model
TRAINED_MODEL_PATH = os.path.join(BASE_DIR, 'runs', 'driver_monitoring', 'weights', 'best.pt')
best_model = YOLO(TRAINED_MODEL_PATH)
print(f'✅ Best model loaded from: {TRAINED_MODEL_PATH}')

# Evaluate on validation set
val_metrics = best_model.val(
    data=DATA_YAML_PATH,
    split='val',
    verbose=True,
    plots=True
)

print(f'\n📊 Validation Metrics:')
print(f'  mAP@0.5:     {val_metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {val_metrics.box.map:.4f}')
print(f'  Precision:   {val_metrics.box.mp:.4f}')
print(f'  Recall:      {val_metrics.box.mr:.4f}')

In [ ]:
# Display training curves
results_dir = os.path.join(BASE_DIR, 'runs', 'driver_monitoring')
results_csv = os.path.join(results_dir, 'results.csv')

if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    plots = [
        ('train/box_loss', 'val/box_loss', 'Box Loss', axes[0,0]),
        ('train/cls_loss', 'val/cls_loss', 'Classification Loss', axes[0,1]),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss', axes[0,2]),
        ('metrics/precision(B)', None, 'Precision', axes[1,0]),
        ('metrics/recall(B)', None, 'Recall', axes[1,1]),
        ('metrics/mAP50(B)', None, 'mAP@0.5', axes[1,2]),
    ]
    
    for train_col, val_col, title, ax in plots:
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], label='Train', color='#3498db', linewidth=2)
        if val_col and val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], label='Val', color='#e74c3c', linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Training & Validation Metrics', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Training results CSV not found. Display after training.')

In [ ]:
# Test inference on test images
test_images_dir = os.path.join(YOLO_DATASET_DIR, 'images', 'test')
test_images = glob.glob(os.path.join(test_images_dir, '*.jpg'))[:8]

if test_images:
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    for ax, img_path in zip(axes.flatten(), test_images):
        result = best_model.predict(img_path, conf=0.4, verbose=False)[0]
        img_annotated = result.plot()
        img_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(os.path.basename(img_path)[:20], fontsize=8)
        ax.axis('off')
    
    plt.suptitle('Model Predictions on Test Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'test_predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Inference on test images complete!')
else:
    print('No test images found.')

## 6. Real-Time Detection (Webcam)

> ⚠️ **Note:** Webcam capture only works on local machine, not in Google Colab.

In [ ]:
# Real-time webcam detection
# Run this cell only on LOCAL machine

CLASS_COLORS_BGR = {
    0: (46, 204, 113),    # Green - Attentive
    1: (243, 156, 18),    # Orange - Distracted
    2: (231, 76, 60),     # Red - Phone
    3: (155, 89, 182),    # Purple - Talking
}

WARNING_CLASSES = {1, 2, 3}  # Classes that trigger warning

def run_realtime_detection(model_path, camera_id=0, conf_threshold=0.5):
    """Run real-time driver monitoring using webcam."""
    model = YOLO(model_path)
    cap = cv2.VideoCapture(camera_id)
    
    if not cap.isOpened():
        print('❌ Cannot open camera!')
        return
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    print('🎥 Real-time detection started. Press Q to quit.')
    
    frame_count = 0
    warning_frames = 0
    WARNING_THRESHOLD = 10  # Frames before showing warning
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        results = model.predict(frame, conf=conf_threshold, verbose=False)[0]
        
        status = 'Attentive'
        status_color = CLASS_COLORS_BGR[0]
        warning = False
        
        for box in results.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = CLASS_NAMES[cls_id]
            color = CLASS_COLORS_BGR[cls_id]
            
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            label_text = f'{label}: {conf:.2f}'
            (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1-th-10), (x1+tw+5, y1), color, -1)
            cv2.putText(frame, label_text, (x1+2, y1-5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            
            if cls_id in WARNING_CLASSES:
                status = label
                status_color = color
                warning = True
        
        if warning:
            warning_frames += 1
        else:
            warning_frames = max(0, warning_frames - 2)
        
        # Overlay status bar
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (frame.shape[1], 60), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
        
        cv2.putText(frame, f'Driver Monitoring System | Status: {status}',
                   (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, status_color, 2)
        
        # Warning flash
        if warning_frames >= WARNING_THRESHOLD:
            cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), (0, 0, 255), 6)
            cv2.putText(frame, '⚠ WARNING! DRIVER DISTRACTED!',
                       (frame.shape[1]//2 - 250, frame.shape[0] - 20),
                       cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
        
        cv2.imshow('Driver Monitoring System - YOLOv8', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print(f'\n📊 Session Summary:')
    print(f'   Total frames processed: {frame_count}')

if not IN_COLAB:
    run_realtime_detection(TRAINED_MODEL_PATH)
else:
    print('ℹ️  Webcam detection skipped (running in Colab).')
    print('   Download the trained model and run locally or via Streamlit/Flask app.')

## 7. Export Model for Deployment

In [ ]:
# Export best model to various formats
print('📦 Exporting trained model...')

# Keep .pt format (default, works with Streamlit/Flask)
shutil.copy(TRAINED_MODEL_PATH, os.path.join(BASE_DIR, 'driver_monitor_best.pt'))
print(f'✅ PyTorch model saved: driver_monitor_best.pt')

# Export to ONNX (for broader compatibility)
try:
    best_model.export(format='onnx', imgsz=640, simplify=True)
    print('✅ ONNX model exported')
except Exception as e:
    print(f'⚠️  ONNX export failed: {e}')

# Model info summary
print('\n📋 Model Summary:')
print(f'   Model path: {os.path.join(BASE_DIR, "driver_monitor_best.pt")}')
print(f'   Classes: {data_yaml["names"]}')
print(f'   Input size: 640x640')
print(f'   Use this .pt file in Streamlit and Flask apps!')

In [ ]:
# Create model card / summary JSON
model_card = {
    'model_name': 'Driver Monitoring System - YOLOv8',
    'base_model': MODEL_SIZE,
    'dataset': 'Driver Gaze RGB Dataset (LISA v1 & v2)',
    'classes': CLASS_NAMES,
    'input_size': 640,
    'epochs_trained': EPOCHS,
    'confidence_threshold': 0.5,
    'model_files': {
        'pytorch': 'driver_monitor_best.pt',
        'onnx': 'driver_monitor_best.onnx'
    },
    'warning_classes': ['Distracted', 'Phone', 'Talking'],
    'safe_class': 'Attentive',
    'project': 'College Project - Not for real vehicle deployment'
}

with open(os.path.join(BASE_DIR, 'model_card.json'), 'w') as f:
    json.dump(model_card, f, indent=2)

print('✅ Model card saved!')
print(json.dumps(model_card, indent=2))

---
## ✅ Project Complete!

| Step | Status | Output |
|------|--------|--------|
| Dataset Download | ✅ | `dataset_raw/` |
| Dataset Preprocessing | ✅ | `yolo_dataset/` |
| Model Training | ✅ | `runs/driver_monitoring/weights/best.pt` |
| Evaluation | ✅ | Metrics logged |
| Export | ✅ | `driver_monitor_best.pt` |

### Next Steps
- Use `driver_monitor_best.pt` in the **Streamlit** or **Flask** application
- See deployment guide for instructions on running and deploying the apps